# 🫀 Transplant Logistics — OpenEnv + GRPO Training

End-to-end notebook: spin up the environment, run a baseline, post-train with GRPO, compare scores.

```
TransplantEnv (HTTP)  →  rollout collector  →  GRPO reward  →  model update
     ↑                                                               ↓
  reset/step/state                                        improved policy
```

**Three tasks:**
- Easy   — single kidney, clear match (baseline ~0.78)
- Medium — liver + heart under viability pressure (baseline ~0.52)
- Hard   — expiry crisis + misleading urgency signal (baseline ~0.31)

## 1. Install dependencies

In [ ]:
%pip install -q trl transformers torch accelerate datasets pydantic fastapi uvicorn httpx openai
print('✓ deps installed')

## 2. Start the environment server

In [ ]:
import subprocess, time, httpx, os, sys

# Clone the repo if running in Colab / fresh environment
if not os.path.exists('transplant-env'):
    # Replace with your HuggingFace Space URL after deploying
    subprocess.run(['git', 'clone',
        'https://huggingface.co/spaces/YOUR_USERNAME/transplant-logistics-env',
        'transplant-env'], check=True)

sys.path.insert(0, 'transplant-env')

# Start server in background
server = subprocess.Popen(
    ['uvicorn', 'server.app:app', '--host', '0.0.0.0', '--port', '7860'],
    cwd='transplant-env',
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

# Wait for it to start
for _ in range(20):
    try:
        r = httpx.get('http://localhost:7860/health', timeout=2)
        if r.status_code == 200:
            print('✓ Server running:', r.json())
            break
    except Exception:
        time.sleep(1)
else:
    print('⚠ Server may not be running — check logs')

## 3. Explore the environment manually

In [ ]:
from server.environment import TransplantEnv, TransplantGrader, TASKS
from models import ActionType, TransplantAction, TransportMode
import json

print('Available tasks:')
for tid, t in TASKS.items():
    print(f"  [{t['difficulty']:6s}] {t['name']}")

# Reset easy task
env = TransplantEnv('task_easy_clear_match')
obs = env.reset(seed=42)

print(f'\nDonors:')
for d in obs.available_donors:
    print(f'  {d.donor_id} | {d.organ_type.value} | blood={d.blood_type.value} | hospital={d.hospital_id} | viability={d.viability_hours}h')

print(f'\nWaitlist:')
for r in obs.waitlist:
    print(f'  {r.recipient_id} | needs={r.organ_needed.value} | blood={r.blood_type.value} | urgency={r.urgency.value} | PRA={r.pra:.0%}')

In [ ]:
# Step through the easy task manually
grader = TransplantGrader()

# Step 1: match D001 → R001 (correct)
r = env.step(TransplantAction(
    action_type=ActionType.MATCH_ORGAN,
    donor_id='D001', recipient_id='R001',
    transport_mode=TransportMode.GROUND
))
print(f'Match: reward={r.reward:.4f}  info={json.dumps(r.info, default=str)[:120]}')

# Step 2: dispatch
r = env.step(TransplantAction(
    action_type=ActionType.DISPATCH_TRANSPORT,
    donor_id='D001', transport_mode=TransportMode.GROUND
))
print(f'Dispatch: reward={r.reward:.4f}')

# Step 3: notify
r = env.step(TransplantAction(action_type=ActionType.NOTIFY_TEAM))
print(f'Notify: reward={r.reward:.4f}  done={r.done}')

# Grade
state = env.state()
grades = grader.grade(state, TASKS['task_easy_clear_match'])
print(f'\nGrades: {grades}')

## 4. Baseline scores (pre-training)

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'sk-...'   # set your key
os.environ['MODEL_NAME']     = 'gpt-4o-mini'

# Run baseline inference
%run transplant-env/baseline/inference.py --quiet

## 5. Load base model for GRPO training

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'   # swap for larger model with more VRAM
device     = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if device == 'cuda' else torch.float32,
    device_map='auto' if device == 'cuda' else None,
)
if device == 'cpu':
    model = model.to(device)

print(f'✓ Loaded {MODEL_NAME}')
print(f'  Params: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M')

## 6. Collect warm-start rollouts

In [ ]:
from training.train_grpo import collect_rollout, build_grpo_dataset, GRADER

task_ids = list(TASKS.keys())

print('Collecting pre-training rollouts...')
pre_scores = {}
for task_id in task_ids:
    rollout = collect_rollout(task_id, model, tokenizer, device, seed=42)
    pre_scores[task_id] = rollout['grade']['aggregate']
    print(f"  {task_id}: {pre_scores[task_id]:.3f} ({rollout['steps']} steps)")

pre_mean = sum(pre_scores.values()) / len(pre_scores)
print(f'\nPre-training mean: {pre_mean:.3f}')

In [ ]:
# Build training dataset from rollouts across all tasks
print('Building GRPO training dataset...')
dataset = build_grpo_dataset(
    task_ids, model, tokenizer, device,
    n_rollouts_per_task=4,   # increase for better training
    seed_start=0,
)
print(f'\nDataset: {len(dataset)} samples')
print(f'Reward distribution: min={min(dataset["reward"]):.3f} '
      f'max={max(dataset["reward"]):.3f} '
      f'mean={sum(dataset["reward"])/len(dataset["reward"]):.3f}')

## 7. GRPO training

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from training.train_grpo import transplant_reward_fn

OUTPUT_DIR = './checkpoints/transplant-grpo'

config = GRPOConfig(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = 3,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    learning_rate               = 5e-6,
    max_new_tokens              = 200,
    temperature                 = 0.7,
    num_generations             = 4,
    beta                        = 0.04,      # KL penalty
    logging_steps               = 10,
    save_steps                  = 100,
    report_to                   = 'none',
    max_prompt_length           = 1024,
    max_completion_length       = 256,
)

trainer = GRPOTrainer(
    model            = model,
    args             = config,
    reward_funcs     = transplant_reward_fn,
    train_dataset    = dataset,
    processing_class = tokenizer,
)

print('Starting GRPO training...')
trainer.train()

## 8. Post-training evaluation

In [ ]:
print('Evaluating trained model...')
post_scores = {}
for task_id in task_ids:
    rollout = collect_rollout(task_id, model, tokenizer, device, seed=999)
    post_scores[task_id] = rollout['grade']['aggregate']

post_mean = sum(post_scores.values()) / len(post_scores)

print(f'\n{"═"*60}')
print('BEFORE vs AFTER GRPO training')
print(f'{"═"*60}')
print(f'{"Task":<45} {"Before":>7} {"After":>7} {"Delta":>7}')
print('-' * 60)
for tid in task_ids:
    before = pre_scores[tid]
    after  = post_scores[tid]
    delta  = after - before
    arrow  = '▲' if delta > 0 else '▼' if delta < 0 else '─'
    print(f'{tid:<45} {before:>7.3f} {after:>7.3f} {arrow}{abs(delta):>6.3f}')
print('-' * 60)
print(f'{"MEAN":<45} {pre_mean:>7.3f} {post_mean:>7.3f} '
      f'{"▲" if post_mean > pre_mean else "▼"}{abs(post_mean - pre_mean):.3f}')

## 9. Save + push to HuggingFace Hub

In [ ]:
import json

# Save model
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'✓ Model saved to {OUTPUT_DIR}')

# Save eval results
results = {
    'base_model': MODEL_NAME,
    'pre_training_scores':  pre_scores,
    'post_training_scores': post_scores,
    'improvement': {tid: post_scores[tid] - pre_scores[tid] for tid in task_ids},
    'mean_pre':  pre_mean,
    'mean_post': post_mean,
}
with open(f'{OUTPUT_DIR}/eval_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('✓ Results saved')
print(json.dumps(results, indent=2))

# Optional: push to Hub
# from huggingface_hub import HfApi
# api = HfApi()
# api.upload_folder(
#     folder_path=OUTPUT_DIR,
#     repo_id='YOUR_USERNAME/transplant-logistics-grpo',
#     repo_type='model',
# )

## 10. Tips for higher scores

| Lever | Effect |
|-------|--------|
| Larger base model (7B+) | Higher baseline and ceiling |
| More rollouts per task (8–16) | Better reward signal coverage |
| Lower temperature during eval | More deterministic, fewer malformed JSON |
| Higher `num_generations` (8) | More contrastive GRPO pairs |
| Lower `beta` (0.01) | More aggressive policy update |
| Curriculum: train easy→hard | Better sample efficiency |
| Add chain-of-thought in prompt | Improves hard task reasoning |